# Data Cleaning and Dataset Splitting

This notebook performs the initial data preparation steps for the credit card fraud detection project.

## Objectives

The main goals of this notebook are:

* Clean and validate the raw dataset
* Prepare the data for machine learning experiments
* Split the dataset into training and testing sets

## Data Source

The raw dataset is located at:

../data/raw/creditcard.csv

Relative path considering that the notebook and the dataset are in different folders in the project root.

## Processing Steps

The following operations are performed in this notebook:

1. Load the raw dataset
2. Validate dataset integrity and basic statistics
3. Perform data cleaning steps
4. Prepare the feature matrix and target variable
5. Split the dataset into **training** and **test** sets

The resulting datasets are saved to:

- ..data/cleaned/'creditcard_cleaned.parquet'
- ..data/splits/train.parquet
- ..data/splits/test.parquet

Relative paths considering that the notebook and the datasets are in different folders in the project root.

## Notes

This notebook was used during the **research phase** of the project to prototype data preparation steps.

The final implementation of the data preparation workflow is included in the **production pipeline** located in the `src/datapipeline` package.


In [29]:
import pandas as pd
from pathlib import Path
from typing import Tuple
from sklearn.model_selection import train_test_split


# Config

In [30]:
target_column = "Class"
dataset_path = '../data/raw/creditcard.csv'  #It is necessary to specify the path to the raw dataset
cleaned_df_path = '../data/cleaned' # location to save the cleaned dataset
test_size = 0.3 # proportion of the dataset to include in the test split
random_state = 42 # Controls the shuffling applied to the data before applying the split
splits_df_path = '../data/splits' # location to save the splitted dataset
min_samples = 1000 # minimun number of samples the dataset must contain
num_classes = 2 # minimum number of target classes the dataset must contain


# Load Data

In [31]:
dataset_path = Path(dataset_path).resolve()
dataset_path

PosixPath('/home/rodolfo/Insync/rodolfopcruz2@gmail.com/Google Drive/Estudo/Projetos-Novos/Credit_card_fraud_detection/data/raw/creditcard.csv')

In [32]:
data = pd.read_csv(dataset_path)

In [33]:
data.describe()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
count,284807.000000,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,...,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,284807.000000,284807.000000
mean,94813.859575,1.168375e-15,3.416908e-16,-1.379537e-15,2.074095e-15,9.604066e-16,1.487313e-15,-5.556467e-16,1.213481e-16,-2.406331e-15,...,1.654067e-16,-3.568593e-16,2.578648e-16,4.473266e-15,5.340915e-16,1.683437e-15,-3.660091e-16,-1.227390e-16,88.349619,0.001727
std,47488.145955,1.958696e+00,1.651309e+00,1.516255e+00,1.415869e+00,1.380247e+00,1.332271e+00,1.237094e+00,1.194353e+00,1.098632e+00,...,7.345240e-01,7.257016e-01,6.244603e-01,6.056471e-01,5.212781e-01,4.822270e-01,4.036325e-01,3.300833e-01,250.120109,0.041527
min,0.000000,-5.640751e+01,-7.271573e+01,-4.832559e+01,-5.683171e+00,-1.137433e+02,-2.616051e+01,-4.355724e+01,-7.321672e+01,-1.343407e+01,...,-3.483038e+01,-1.093314e+01,-4.480774e+01,-2.836627e+00,-1.029540e+01,-2.604551e+00,-2.256568e+01,-1.543008e+01,0.000000,0.000000
25%,54201.500000,-9.203734e-01,-5.985499e-01,-8.903648e-01,-8.486401e-01,-6.915971e-01,-7.682956e-01,-5.540759e-01,-2.086297e-01,-6.430976e-01,...,-2.283949e-01,-5.423504e-01,-1.618463e-01,-3.545861e-01,-3.171451e-01,-3.269839e-01,-7.083953e-02,-5.295979e-02,5.600000,0.000000
50%,84692.000000,1.810880e-02,6.548556e-02,1.798463e-01,-1.984653e-02,-5.433583e-02,-2.741871e-01,4.010308e-02,2.235804e-02,-5.142873e-02,...,-2.945017e-02,6.781943e-03,-1.119293e-02,4.097606e-02,1.659350e-02,-5.213911e-02,1.342146e-03,1.124383e-02,22.000000,0.000000
75%,139320.500000,1.315642e+00,8.037239e-01,1.027196e+00,7.433413e-01,6.119264e-01,3.985649e-01,5.704361e-01,3.273459e-01,5.971390e-01,...,1.863772e-01,5.285536e-01,1.476421e-01,4.395266e-01,3.507156e-01,2.409522e-01,9.104512e-02,7.827995e-02,77.165000,0.000000
max,172792.000000,2.454930e+00,2.205773e+01,9.382558e+00,1.687534e+01,3.480167e+01,7.330163e+01,1.205895e+02,2.000721e+01,1.559499e+01,...,2.720284e+01,1.050309e+01,2.252841e+01,4.584549e+00,7.519589e+00,3.517346e+00,3.161220e+01,3.384781e+01,25691.160000,1.000000


# Validate Data

In [34]:
def validate_data(
        df: pd.DataFrame,
        target_column: str,
        min_samples: int = 1000,
        num_classes: int = 2,
) -> None:
    
    """
    Validates if the dataset is valid

    Args:
        df (pd.DataFrame): pandas dataframe after preliminary cleaning
        target_column (str): target column
        min_samples (int, optional): required min number of samples. Defaults to 1000.
        num_classes (int, optional): number of classes in the target. Defaults to 2.
        logger (logging.Logger, optional): Logger instance.
  
    Raises:
        ValueError: the number of samples is less than min_samples
        ValueError: the number of classes is not equal to num_classes
        ValueError: one target class has no samples
    """

    print('Validating Data')
    
    if df.shape[0] < min_samples:
        raise ValueError(f"Dataset must have at least {min_samples} samples")
    
    if target_column not in df.columns:
        raise ValueError(f"Target column '{target_column}' not found")

    class_count = df[target_column].value_counts(dropna=True)

    if len(class_count) != num_classes:
        raise ValueError(f"Expected {num_classes} classes, found {len(class_count)}")
    
    if class_count.min() <= 0:
     raise ValueError("One target class has no samples")
    print('Dataset Validated')

In [35]:
validate_data(
        df = data,
        target_column = target_column,
        min_samples = min_samples,
        num_classes =num_classes)


Validating Data
Dataset Validated


# Data Cleaning

In [36]:
# Path to save the clened dataframe
cleaned_df_path = Path(cleaned_df_path).resolve()
cleaned_df_path.mkdir(parents=True, exist_ok=True)


In [37]:
def clean_data(df: pd.DataFrame, 
               target_column: str, 
) -> Tuple[pd.DataFrame, int]:

    """
    Perform preliminary data cleaning.

    Steps:
    - Remove duplicate rows
    - Remove rows without target
    - Sanity check on target values

    Args:
        df (pd.DataFrame): Input dataframe.
        target_column (str): Name of target column.
        logger (logging.Logger, optional): Logger instance.

    Returns:
        Tuple[pd.DataFrame, int]: Cleaned dataframe and number of rows removed.
    """

    initial_rows = df.shape[0]

    # Remove duplicates
    df = df.drop_duplicates()
    duplicated_rows_removed = initial_rows - df.shape[0]
    
    print(f'Removed {duplicated_rows_removed} duplicated rows')


    # Remove rows without target
    before = df.shape[0]
    df = df.dropna(subset=[target_column])
    removed_missing_target = before - df.shape[0]
    
    print(f'Removed {removed_missing_target} rows without target')

    # Sanity checks
    if (df[target_column] < 0).any():
        raise ValueError("Invalid target values detected")

    total_removed = duplicated_rows_removed + removed_missing_target
    print(f'Total rows removed: {total_removed}')

    return df, total_removed



In [38]:
df_cleaned, _ = clean_data(data,
                       target_column)


Removed 1081 duplicated rows
Removed 0 rows without target
Total rows removed: 1081


In [39]:
df_cleaned.to_parquet(cleaned_df_path / 'creditcard_cleaned.parquet')

# Data Splitting

In [40]:
#Path to save the training and testing dataframes
splits_df_path = Path(splits_df_path).resolve()
splits_df_path.mkdir(parents=True, exist_ok=True)


In [41]:
def split_data(df: pd.DataFrame,
               target_column: str,
               test_size: float,
               random_state: int,
               ) -> Tuple[pd.DataFrame, pd.DataFrame]:

    """
    Split a DataFrame into training and testing sets.

    Args:
        df (pd.DataFrame): The DataFrame to split.
        target_column (str): The name of the target column.
        test_size (float): The proportion of the dataset to include in the test split.
        random_state (int): The seed used by the random number generator.
        logger (logging.Logger | None, optional): The logger to use. Defaults to None.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: A tuple containing the training and testing DataFrames.
    """

    X = df.drop(target_column, axis=1)
    y = df[target_column]

    print('Spliting data...')

    if y.nunique() < 2:
        raise ValueError("Target column must have at least two classes for stratified split")

        
    X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                        test_size=test_size, 
                                                        random_state=random_state,
                                                        stratify=y)
    print(f'{df.shape[0]} samples split into {X_train.shape[0]} ' 
                    f'train samples and {X_test.shape[0]} test samples')
    print(f'{X_train.shape[0]/df.shape[0]*100:.2f}% of the '
                    f'dataset is used for training and {X_test.shape[0]/df.shape[0]*100:.2f}% for testing' )

    df_train = X_train
    df_test  = X_test
    df_train[target_column] = y_train
    df_test[target_column] = y_test

    return df_train, df_test

In [42]:
X_train, X_test = split_data(df = df_cleaned, 
                            target_column = target_column,
                            test_size = test_size,
                            random_state = random_state)

Spliting data...
283726 samples split into 198608 train samples and 85118 test samples
70.00% of the dataset is used for training and 30.00% for testing


In [43]:
X_train.to_parquet(splits_df_path / 'train.parquet')
X_test.to_parquet(splits_df_path / 'test.parquet')